**Feature Selection Method:** FLAML   
**Dataset:** QUT-DV25 (Filetop)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:

gc.collect()
tf.keras.backend.clear_session()

In [ ]:
data.head()

,Package_Name,Total_Reads,Total_Writes,Total_Read_Data_Transfer,Total_Write_Data_Transfer,Read_Processes,Write_Processes,Read_Data_Transfer_Processes,Write_Data_Transfer_Processes,File_Access_Processes,Level
0,10Cent10-999.0.4.tar.gz,1345126,711470,273890605,580106,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...","StreamTrans, strace, Classif~date, pip, sqldb:...","gnome-system-mo, pgrep, pip, StreamTrans, sqld...",1
1,10Cent11-999.0.4.tar.gz,1192399,585508,251798360,547692,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...","StreamTrans, strace, Classif~date, pip, sqldb:...","gnome-system-mo, pgrep, pip, sqldb:c~lite, Str...",1
2,11Cent-999.0.0.tar.gz,1083052,648800,233980925,463210,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...","StreamTrans, strace, Classif~date, pip, sqldb:...","gnome-system-mo, pip, pgrep, sqldb:c~lite, Str...",1
3,11Cent-999.0.1.tar.gz,1010263,560585,211948676,448241,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...","StreamTrans, strace, Classif~date, pip, sqldb:...","gnome-system-mo, pip, pgrep, sqldb:c~lite, Str...",1
4,11Cent-999.0.2.tar.gz,928456,520884,201673655,441561,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...","StreamTrans, strace, Classif~date, pip, sqldb:...","gnome-system-mo, pip, pgrep, sqldb:c~lite, Str...",1


In [ ]:
if 'Package_Name' in data.columns:
    data = data.drop(columns=['Package_Name'])

# Combine categorical columns into a single text feature
categorical_columns = [col for col in data.columns if data[col].dtype == 'object' and col != 'Level']
data['Combined_Categorical'] = data[categorical_columns].fillna('').astype(str).agg(' '.join, axis=1)

# Vectorize the combined categorical text column using n-grams
vectorizer = CountVectorizer(ngram_range=(2, 3))
categorical_ngrams = vectorizer.fit_transform(data['Combined_Categorical'])

# Ensure only valid numeric columns are selected
numerical_columns = [
    col for col in data.columns
    if pd.to_numeric(data[col], errors='coerce').notnull().all() and col != 'Level'
]

# Prepare numerical features
numerical_features = data[numerical_columns].fillna(0).astype(float)

# Convert numerical features to sparse matrix
numerical_features_sparse = csr_matrix(numerical_features.values)

# Combine numerical features with n-gram features
X = hstack([numerical_features_sparse, categorical_ngrams])

# Scale features before combining
scaler = StandardScaler(with_mean=False)  # Sparse matrices need `with_mean=False`
X_scaled = scaler.fit_transform(X)

In [ ]:

X = data.drop(columns=['Level'])
y = data['Level']

if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize FLAML
automl = AutoML()
automl_settings = {
    "time_budget": 300,
    "metric": "accuracy",
    "task": "classification",
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# Feature importance & selection
selected_features = X.columns.tolist()
dropped_features = []

try:
    importances = automl.model.feature_importances_

    # Sometimes LGBM may drop unused features -> align with all columns
    if len(importances) < X.shape[1]:
        full_importances = np.zeros(X.shape[1])
        # Use feature names from model if available
        try:
            model_features = automl.model.booster_.feature_name()
            for i, col in enumerate(X.columns):
                if col in model_features:
                    idx = model_features.index(col)
                    full_importances[i] = importances[idx]
        except:
            full_importances[:len(importances)] = importances
        importances = full_importances

    varimp = pd.DataFrame({'variable': X.columns, 'importance': importances})
    varimp['relative_importance'] = varimp['importance'] / varimp['importance'].sum()

    threshold = 0.03
    selected_features = list(varimp[varimp['relative_importance'] > threshold]['variable'])
    dropped_features = list(varimp[varimp['relative_importance'] <= threshold]['variable'])

    print("Feature importance table:\n", varimp)

except AttributeError:
    print("Feature importance not available for the chosen model.")

print("\nSelected features:", selected_features)
print("Number of features kept:", len(selected_features))
print("\nDropped features:", dropped_features)
print("Number of features dropped:", len(dropped_features))


[flaml.automl.logger: 08-29 20:38:54] {1680} INFO - task = classification
[flaml.automl.logger: 08-29 20:38:54] {1691} INFO - Evaluation method: cv
[flaml.automl.logger: 08-29 20:38:54] {1789} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 08-29 20:38:54] {1901} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'lrl1']
[flaml.automl.logger: 08-29 20:38:54] {2219} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 08-29 20:38:55] {2346} INFO - Estimated sufficient time budget=2735s. Estimated necessary time budget=63s.
[flaml.automl.logger: 08-29 20:38:55] {2398} INFO -  at 0.4s,	estimator lgbm's best error=0.2025,	best estimator lgbm's best error=0.2025
[flaml.automl.logger: 08-29 20:38:55] {2219} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 08-29 20:38:55] {2398} INFO -  at 0.6s,	estimator lgbm's best error=0.2005,	best estimator lgbm's best error=0.2005
[flaml.automl.logger: 08-29 20:38:

[flaml.automl.logger: 08-29 20:39:31] {2219} INFO - iteration 34, current learner lgbm
[flaml.automl.logger: 08-29 20:39:33] {2398} INFO -  at 39.3s,	estimator lgbm's best error=0.0766,	best estimator lgbm's best error=0.0766
[flaml.automl.logger: 08-29 20:39:33] {2219} INFO - iteration 35, current learner rf
[flaml.automl.logger: 08-29 20:39:35] {2398} INFO -  at 41.1s,	estimator rf's best error=0.1700,	best estimator lgbm's best error=0.0766
[flaml.automl.logger: 08-29 20:39:35] {2219} INFO - iteration 36, current learner xgboost
[flaml.automl.logger: 08-29 20:39:35] {2398} INFO -  at 41.2s,	estimator xgboost's best error=0.1554,	best estimator lgbm's best error=0.0766
[flaml.automl.logger: 08-29 20:39:35] {2219} INFO - iteration 37, current learner rf
[flaml.automl.logger: 08-29 20:39:37] {2398} INFO -  at 42.6s,	estimator rf's best error=0.1628,	best estimator lgbm's best error=0.0766
[flaml.automl.logger: 08-29 20:39:37] {2219} INFO - iteration 38, current learner xgboost
[flaml.a

[flaml.automl.logger: 08-29 20:40:20] {2219} INFO - iteration 69, current learner lgbm
[flaml.automl.logger: 08-29 20:40:23] {2398} INFO -  at 88.8s,	estimator lgbm's best error=0.0720,	best estimator xgb_limitdepth's best error=0.0684
[flaml.automl.logger: 08-29 20:40:23] {2219} INFO - iteration 70, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:40:26] {2398} INFO -  at 91.4s,	estimator xgb_limitdepth's best error=0.0635,	best estimator xgb_limitdepth's best error=0.0635
[flaml.automl.logger: 08-29 20:40:26] {2219} INFO - iteration 71, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:40:27] {2398} INFO -  at 92.7s,	estimator xgb_limitdepth's best error=0.0635,	best estimator xgb_limitdepth's best error=0.0635
[flaml.automl.logger: 08-29 20:40:27] {2219} INFO - iteration 72, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:40:32] {2398} INFO -  at 97.5s,	estimator xgb_limitdepth's best error=0.0603,	best estimator xgb_limitdepth's best error=0.0

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 20:41:42] {2398} INFO -  at 167.5s,	estimator lrl1's best error=0.2419,	best estimator xgb_limitdepth's best error=0.0584
[flaml.automl.logger: 08-29 20:41:42] {2219} INFO - iteration 85, current learner lrl1


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 20:41:43] {2398} INFO -  at 168.6s,	estimator lrl1's best error=0.2418,	best estimator xgb_limitdepth's best error=0.0584
[flaml.automl.logger: 08-29 20:41:43] {2219} INFO - iteration 86, current learner xgb_limitdepth


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_sag.py:354: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  ConvergenceWarning,


[flaml.automl.logger: 08-29 20:41:48] {2398} INFO -  at 174.2s,	estimator xgb_limitdepth's best error=0.0584,	best estimator xgb_limitdepth's best error=0.0584
[flaml.automl.logger: 08-29 20:41:48] {2219} INFO - iteration 87, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:42:07] {2398} INFO -  at 192.9s,	estimator xgb_limitdepth's best error=0.0584,	best estimator xgb_limitdepth's best error=0.0584
[flaml.automl.logger: 08-29 20:42:07] {2219} INFO - iteration 88, current learner xgb_limitdepth
[flaml.automl.logger: 08-29 20:42:10] {2398} INFO -  at 195.6s,	estimator xgb_limitdepth's best error=0.0584,	best estimator xgb_limitdepth's best error=0.0584
[flaml.automl.logger: 08-29 20:42:10] {2219} INFO - iteration 89, current learner xgboost
[flaml.automl.logger: 08-29 20:42:13] {2398} INFO -  at 198.5s,	estimator xgboost's best error=0.0996,	best estimator xgb_limitdepth's best error=0.0584
[flaml.automl.logger: 08-29 20:42:13] {2219} INFO - iteration 90, current learner r

In [ ]:
selected_features = ['Total_Reads', 'Total_Writes', 'Total_Read_Data_Transfer', 'Total_Write_Data_Transfer', 'Read_Processes', 'Write_Processes', 'Read_Data_Transfer_Processes', 'Write_Data_Transfer_Processes', 'File_Access_Processes']